# EDA Data VCT 2023–2026
Kerangka ini mengikuti delapan butir **Bagian C (Rencana EDA)** di proposal. Jalankan sel dari atas ke bawah.

Cara pakai:
- Letakkan notebook ini di folder yang berisi `vct_2023` ... `vct_2026` (atau ubah `BASE`).
- Setiap bagian punya sel **Catatan kita** di akhirnya. Tulis temuan dan interpretasi kalian sendiri di sana, karena itu yang akan dibaca dosen.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path(".")                      # folder yang berisi vct_2021 ... vct_2026
YEARS = [2023, 2024, 2025, 2026]      # data utama proposal
KEY = ['Tournament', 'Stage', 'Match Type', 'Match Name', 'Map', 'Team']   # 1 komposisi = 1 tim di 1 map

def read(year, sub, name):
    path = BASE / f"vct_{year}" / sub / name
    return pd.read_csv(path, keep_default_na=False, na_values=[''], low_memory=False)

## C.i Profiling dasar

In [ ]:
frames = []
for y in YEARS:
    o = read(y, "matches", "overview.csv")
    o['Tahun'] = y
    frames.append(o)
overview = pd.concat(frames, ignore_index=True)

print("bentuk data:", overview.shape)
overview.head()

In [ ]:
overview.info()

In [ ]:
# 10 kolom dengan nilai kosong terbanyak
overview.isna().sum().sort_values(ascending=False).head(10)

### Membentuk tabel komposisi
Dari `overview.csv` kita ambil baris `Side == 'both'` dan buang baris ringkasan `All Maps`. Satu komposisi = satu tim pada satu map, dan dianggap valid bila berisi tepat 5 pemain.

In [ ]:
pemain = overview[(overview['Side'] == 'both') & (overview['Map'] != 'All Maps')].copy()
pemain['Agents'] = pemain['Agents'].astype(str)

comp = (pemain.groupby(KEY + ['Tahun'])
        .agg(jumlah_pemain=('Player', 'size'), agents=('Agents', list))
        .reset_index())

print("semua komposisi:", len(comp))
print(comp['jumlah_pemain'].value_counts())

n_tidak_valid = int((comp['jumlah_pemain'] != 5).sum())
comp = comp[comp['jumlah_pemain'] == 5].reset_index(drop=True)   # hanya komposisi valid
pick = pemain.merge(comp[KEY], on=KEY)                           # 1 baris = 1 pemain/agent pada komposisi valid

print("komposisi valid:", len(comp), "| baris pemain:", len(pick))
assert len(pick) == len(comp) * 5

**Catatan kita (C.i):** ...

## C.ii Distribusi kelas (frekuensi pemilihan tiap agent)

In [ ]:
dist = pick['Agents'].value_counts()
tabel = pd.DataFrame({'jumlah': dist, 'persen': (dist / dist.sum() * 100).round(1)})
print(tabel.head(10))

dist.plot(kind='bar', figsize=(12, 4), title='Frekuensi pemilihan agent 2023-2026')
plt.ylabel('jumlah dipilih'); plt.tight_layout(); plt.show()

print("jumlah agent:", len(dist))
print("rasio agent terbanyak : tersedikit =", round(dist.max() / dist.min(), 1))

**Catatan kita (C.ii):** apakah kelasnya seimbang? agent mana yang sangat jarang dipilih, dan kenapa (misalnya agent baru)?

## C.iii Distribusi map

In [ ]:
game = comp.drop_duplicates(['Tournament', 'Stage', 'Match Type', 'Match Name', 'Map'])   # 1 baris = 1 map yang dimainkan
map_count = game['Map'].value_counts()
print(map_count)

map_count.plot(kind='bar', figsize=(8, 3.5), title='Jumlah map yang dimainkan 2023-2026')
plt.tight_layout(); plt.show()

**Catatan kita (C.iii):** map mana yang paling sering dan paling jarang dimainkan? apakah ada map yang jumlah datanya sangat sedikit?

## C.iv Agent per map (pick rate dan win rate)
**Pick rate** = persentase komposisi di map itu yang memuat agent tersebut.

In [ ]:
per_map = pick.groupby(['Map', 'Agents']).size().rename('jumlah').reset_index()
total_map = comp.groupby('Map').size().rename('total_komposisi')
per_map = per_map.merge(total_map, on='Map')
per_map['pick_rate'] = per_map['jumlah'] / per_map['total_komposisi']

pr = per_map.pivot(index='Agents', columns='Map', values='pick_rate').fillna(0)

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(pr.values, aspect='auto')
ax.set_xticks(range(len(pr.columns))); ax.set_xticklabels(pr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(pr.index))); ax.set_yticklabels(pr.index)
ax.set_title('Pick rate agent per map (2023-2026)')
fig.colorbar(im); plt.tight_layout(); plt.show()

**Win rate** dihitung dari `teams_picked_agents.csv`. File ini berisi baris rekap (`All Stages`, `All Match Types`) yang harus dibuang, kalau tidak jumlahnya terhitung dua kali.

In [ ]:
tpa = pd.concat([read(y, "agents", "teams_picked_agents.csv") for y in YEARS], ignore_index=True)
print("baris sebelum filter:", len(tpa), "| total Maps Played:", tpa['Total Maps Played'].sum())

tpa = tpa[(tpa['Stage'] != 'All Stages') & (tpa['Match Type'] != 'All Match Types') & (tpa['Map'] != 'All Maps')]
print("baris sesudah filter:", len(tpa), "| total Maps Played:", tpa['Total Maps Played'].sum())
print("pembanding: jumlah baris pemain dari overview =", len(pick))   # harus mendekati

wr = tpa.groupby(['Map', 'Agent'])[['Total Wins By Map', 'Total Loss By Map']].sum()
wr['dimainkan'] = wr.sum(axis=1)
wr['win_rate'] = wr['Total Wins By Map'] / wr['dimainkan']
wr = wr.reset_index()

MIN_MATCH = 20      # batas minimum sampel; boleh diubah
print("kombinasi agent-map:", len(wr), "| yang dimainkan >=", MIN_MATCH, "kali:", (wr['dimainkan'] >= MIN_MATCH).sum())
wr[wr['dimainkan'] >= MIN_MATCH].sort_values('win_rate', ascending=False).head(10)

In [ ]:
wr_ok = wr[wr['dimainkan'] >= MIN_MATCH]
wr_pivot = wr_ok.pivot(index='Agent', columns='Map', values='win_rate')

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(wr_pivot.values, aspect='auto')
ax.set_xticks(range(len(wr_pivot.columns))); ax.set_xticklabels(wr_pivot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(wr_pivot.index))); ax.set_yticklabels(wr_pivot.index)
ax.set_title(f'Win rate agent per map (minimal {MIN_MATCH} kali dimainkan)')
fig.colorbar(im); plt.tight_layout(); plt.show()

**Catatan kita (C.iv):** agent mana yang dominan di map tertentu? apakah win rate yang tinggi didukung sampel yang cukup banyak?

## C.v Peran agent
Dataset **tidak punya kolom peran**, jadi tabelnya dibuat manual. Isian di bawah adalah tebakan awal.
**Wajib diverifikasi dari sumber resmi Valorant** sebelum dipakai. Agent yang belum diketahui perannya dibiarkan `None` (isi sendiri).

In [ ]:
ROLE = {
    # Duelist
    'jett': 'Duelist', 'phoenix': 'Duelist', 'reyna': 'Duelist', 'raze': 'Duelist',
    'yoru': 'Duelist', 'neon': 'Duelist', 'iso': 'Duelist', 'waylay': 'Duelist',
    # Initiator
    'sova': 'Initiator', 'breach': 'Initiator', 'skye': 'Initiator', 'kayo': 'Initiator',
    'fade': 'Initiator', 'gekko': 'Initiator', 'tejo': 'Initiator',
    # Controller
    'brimstone': 'Controller', 'omen': 'Controller', 'viper': 'Controller',
    'astra': 'Controller', 'harbor': 'Controller', 'clove': 'Controller',
    # Sentinel
    'sage': 'Sentinel', 'cypher': 'Sentinel', 'killjoy': 'Sentinel',
    'chamber': 'Sentinel', 'deadlock': 'Sentinel', 'vyse': 'Sentinel',
    # agent baru: isi sendiri setelah dicek
    'miks': None, 'veto': None,
}

agents_data = sorted(pick['Agents'].unique())
print("agent di data:", len(agents_data))
print("agent yang belum punya peran:", [a for a in agents_data if ROLE.get(a) is None])

In [ ]:
pick['Peran'] = pick['Agents'].map(ROLE)

# hanya komposisi yang kelima agent-nya sudah punya peran
lengkap = pick.groupby(KEY)['Peran'].apply(lambda s: s.notna().all())
role_count = pick.groupby(KEY)['Peran'].value_counts().unstack(fill_value=0)
role_count = role_count[lengkap.reindex(role_count.index).fillna(False)]
print("komposisi yang dipakai:", len(role_count), "dari", len(comp))

for p in role_count.columns:
    print(p, "-> persentase komposisi menurut jumlah:", role_count[p].value_counts(normalize=True).sort_index().round(3).to_dict())

print()
print("komposisi dengan minimal 1 Controller:", round((role_count['Controller'] >= 1).mean() * 100, 1), "%")
print("komposisi dengan >= 3 Duelist       :", round((role_count['Duelist'] >= 3).mean() * 100, 1), "%")

**Catatan kita (C.v):** dari angka di atas, apa batas jumlah tiap peran yang masuk akal untuk aturan keseimbangan peran? (misalnya maksimal berapa Duelist, minimal berapa Controller)

## C.vi Tren waktu (pergeseran meta)

In [ ]:
per_tahun = pick.groupby(['Tahun', 'Agents']).size().rename('jumlah').reset_index()
total_tahun = comp.groupby('Tahun').size().rename('total_komposisi')
per_tahun = per_tahun.merge(total_tahun, on='Tahun')
per_tahun['pick_rate'] = per_tahun['jumlah'] / per_tahun['total_komposisi']

for y in YEARS:
    top = per_tahun[per_tahun['Tahun'] == y].nlargest(5, 'pick_rate')
    print(y, [(a, round(p * 100, 1)) for a, p in zip(top['Agents'], top['pick_rate'])])

tren = per_tahun.pivot(index='Tahun', columns='Agents', values='pick_rate').fillna(0)
tren[dist.head(8).index].plot(figsize=(9, 4), marker='o', title='Pick rate 8 agent terpopuler per tahun')
plt.ylabel('pick rate'); plt.tight_layout(); plt.show()

print("jumlah agent per tahun:", per_tahun.groupby('Tahun')['Agents'].nunique().to_dict())

**Catatan kita (C.vi):** agent mana yang naik atau turun? bagaimana pengaruhnya terhadap rencana pembagian training/testing berdasarkan tahun?

## C.vii Kualitas data

In [ ]:
print("baris duplikat di overview   :", overview.duplicated().sum())
print("pemain bernama 'nan'          :", (overview['Player'] == 'nan').sum())
print("baris tim 'TBD'               :", (overview['Team'] == 'TBD').sum())
print("komposisi tidak berisi 5 pemain:", n_tidak_valid)
print("sel Agents berisi >1 agent    :", pemain['Agents'].str.contains(',').sum())
print("Agents kosong                 :", (pemain['Agents'] == 'nan').sum())

In [ ]:
# sampel per kombinasi agent-map (untuk menilai apakah win rate bisa dipercaya)
sampel = wr['dimainkan']
print(sampel.describe().round(1))
print(f"kombinasi dengan sampel < {MIN_MATCH}: {(sampel < MIN_MATCH).sum()} dari {len(sampel)}")

**Catatan kita (C.vii):** masalah data apa yang masih ada, dan bagaimana kita menanganinya di praproses?

## Menyimpan hasil
Tabel komposisi bersih ini akan dipakai di praproses dan model. Folder `hasil/` sebaiknya masuk `.gitignore` bila ukurannya besar.

In [ ]:
Path("hasil").mkdir(exist_ok=True)

simpan = comp.copy()
simpan['agents'] = simpan['agents'].apply(lambda l: '|'.join(l))
simpan.to_csv("hasil/komposisi_valid_2023_2026.csv", index=False)
wr.to_csv("hasil/winrate_agent_per_map.csv", index=False)
print("tersimpan:", list(Path("hasil").glob("*.csv")))

## Kesimpulan EDA
Tulis dengan kata kalian sendiri:
1. Seberapa tidak seimbang kelas agent, dan apa dampaknya pada model?
2. Map dan agent mana yang paling berpengaruh?
3. Aturan keseimbangan peran apa yang kita ambil dari data?
4. Apa keputusan cakupan data akhir (tahun yang dipakai)?